In [0]:
# Databricks Notebook: Nb_Audit_Success
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime

# Define parameters passed from ADF
dbutils.widgets.text("pipeline_name", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("trigger_type", "")
dbutils.widgets.text("trigger_name", "")
dbutils.widgets.text("source_schema_name", "")
dbutils.widgets.text("source_table_name", "")
dbutils.widgets.text("target_catalog_name", "bankingpoc")
dbutils.widgets.text("target_schema_name", "bronze")
dbutils.widgets.text("target_table_name", "")
dbutils.widgets.text("copy_duration", "0")
dbutils.widgets.text("pipeline_start_time", "")
dbutils.widgets.text("rows_read", "0")
dbutils.widgets.text("rows_copied", "0")
dbutils.widgets.text("data_read", "0")
dbutils.widgets.text("data_written", "0")
dbutils.widgets.text("load_type", "")
dbutils.widgets.text("adls_target_path", "")

pipeline_name = dbutils.widgets.get("pipeline_name")
run_id = dbutils.widgets.get("run_id")
trigger_type = dbutils.widgets.get("trigger_type")
trigger_name = dbutils.widgets.get("trigger_name")
source_schema = dbutils.widgets.get("source_schema_name")
source_table = dbutils.widgets.get("source_table_name")
target_catalog = dbutils.widgets.get("target_catalog_name")
target_schema = dbutils.widgets.get("target_schema_name")
target_table = dbutils.widgets.get("target_table_name")
copy_duration = dbutils.widgets.get("copy_duration")
pipeline_start_time = dbutils.widgets.get("pipeline_start_time")
rows_read = dbutils.widgets.get("rows_read")
rows_copied = dbutils.widgets.get("rows_copied")
data_read = dbutils.widgets.get("data_read")
data_written = dbutils.widgets.get("data_written")
load_type = dbutils.widgets.get("load_type")
adls_target_path = dbutils.widgets.get("adls_target_path")
current_time_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# 1. Insert Successful Audit Record
audit_df = spark.createDataFrame([(
    pipeline_name, "SQL_SERVER", run_id, trigger_type, trigger_name,
    source_schema, source_table, target_catalog, target_schema, target_table,
    copy_duration, pipeline_start_time, current_time_str, "SUCCESS", None,
    rows_read, rows_copied, data_read, data_written, load_type,
    pipeline_start_time, current_time_str, current_time_str, adls_target_path
)], schema="""
    pipeline_name STRING,
    source_name STRING,
    run_id STRING,
    trigger_type STRING,
    trigger_name STRING,
    source_schema_name STRING,
    source_table_name STRING,
    target_catalog_name STRING,
    target_schema_name STRING,
    target_table_name STRING,
    copy_duration STRING,
    pipeline_start_time STRING,
    pipeline_end_time STRING,
    execution_status STRING,
    error_details STRING,
    rows_read STRING,
    rows_copied STRING,
    data_read STRING,
    data_written STRING,
    load_type STRING,
    start_time STRING,
    end_time STRING,
    last_load_date STRING,
    adls_target_path STRING
""")

audit_df.write.format("delta").mode("append").saveAsTable("bankingpoc.audit.pipeline_run_log")

# 2. Advance Watermark in control.pipeline_config for Incremental loads
if load_type.lower() == "incremental":
    spark.sql(f"""
        UPDATE bankingpoc.control.pipeline_config
        SET last_pipeline_run_date = '{pipeline_start_time}',
            last_utc_end_time = '{current_time_str}'
        WHERE source_table_name = '{source_table}'
    """)

dbutils.notebook.exit("SUCCESS_LOGGED")